# Evaluation, legal domain (Kaggle)

Runs every metric of `docs/evaluation_metrics.md` for the **legal** domain: 5 single-question
metrics, 2 multi-question metrics, latency, and the end-to-end check through OCR. Each is
measured for **no privacy** (standard RAG) and **ours** (Agents 1-3 + the session monitor).
Step-by-step guide: `docs/evaluation_metrics.md`, section "Running it".

**Before running** (right sidebar):
* **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
* **Add-ons -> Secrets**: `GEMINI_API_KEY`, optionally `GEMINI_API_KEY_2` ... `GEMINI_API_KEY_9` (used for labelling the scans, and by the legal profile wherever it calls Gemini). Tick each as attached to this notebook.

**To let it run on its own** (recommended): with `SCALE = "5h"` in cell 1, click **Save Version
-> Save & Run All (Commit) -> Save** (top right) and close Kaggle. Every cell runs in the
background; when it finishes, the version's **Output** tab holds `eval_results_legal_5h.zip`.

| Cell | What it does | Time at `SCALE = "5h"` |
|---|---|---|
| 1 Setup | clone, install, Ollama and the models, keys | ~15 min (every session) |
| 2 Data | download the real data, copy the generated corpus, write the questions | ~3 min |
| 3 OCR | read 25 real scans (end-to-end check) | ~20 min |
| 4 Labels | Gemini marks private values and writes questions for the 25 scans | ~5 min |
| 5 Build | Agents 1-3 over ~350 documents (150 generated, 150 real, 2 x 25 scan parts) | ~2 h |
| 6 Single | 100 Target Info, 100 repeat, 200 membership and 100 utility questions | ~1 h |
| 7 Multi | 60 attack sessions (20 targets x 3 protocols) x 8 rounds | ~45 min |
| 8 End to end | the scans' questions on both stores | ~10 min |
| 9 Report | every number, REPORT.md and the figures | ~3 min |
| 10 Pack | `eval_results_legal_5h.zip` to download and commit | ~1 min |

`SCALE = "5h"` (the default) fits in about 4.5 hours. Optionally run `SCALE = "quick"` first
(about 10 minutes, a few items per step) to check the set-up; `SCALE = "full"` is the plan's
full volumes (12-15 hours). Each scale keeps its results in its own folder. Run interactively,
a session stops when the tab closes: run cell 1 again and carry on from the first unfinished
cell, since every step skips what is already on disk.

In [ ]:
# 1 Setup (run at the start of every session)
SCALE = "5h"   # "5h": every metric on less data, ~4.5 h; "quick": ~10 min check; "full": the plan's volumes (~12-15 h)
LEGAL_BRANCH = "law-pipeline-final"   # the branch with the legal agents
ANSWERS = "ollama"   # "ollama": the legal profile's Ollama model answers (fits the time);
                     # "gemini": the profile's Gemini answer model (needs many keys and days)
AGENT1_ADAPTER = ""  # optional: path of a fine-tuned Agent 1 LoRA adapter (else the base model)
OCR_MODEL = "qwen3-vl:8b-instruct"    # reads the scans (end-to-end check; main's OCR)

import json, os, subprocess, sys, time, urllib.request
from pathlib import Path

W = Path("/kaggle/working")
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
MAIN = W / "main"      # this evaluation suite and the OCR (step 0) come from main
LEGAL = W / "legal"    # the legal pipeline (Agents 1-3, monitor, scorer) comes from the branch
for path, branch in ((MAIN, "main"), (LEGAL, LEGAL_BRANCH)):
    if path.exists():
        subprocess.run(["git", "-C", str(path), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", branch, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(path)], check=True)
!git -C /kaggle/working/main log --oneline -1
!git -C /kaggle/working/legal log --oneline -1
os.chdir(LEGAL)
!pip install -q -r /kaggle/working/main/requirements.txt
!pip install -q -U transformers accelerate peft bitsandbytes google-genai presidio-analyzer sentence-transformers chromadb rank_bm25 rapidfuzz rouge-score matplotlib
!pip install -q -e /kaggle/working/legal
!python -m spacy download en_core_web_lg -q

# Ollama on both GPUs (on GPU 0 only when a fine-tuned Agent 1 takes GPU 1)
!apt-get update -qq && apt-get install -y -qq zstd
!which ollama || (curl -fsSL https://ollama.com/install.sh | sh)
def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); return True
    except Exception:
        return False
if not ollama_up():
    env = {**os.environ, **({"CUDA_VISIBLE_DEVICES": "0"} if AGENT1_ADAPTER else {})}
    subprocess.Popen(["ollama", "serve"], stdout=open(W / "ollama.log", "a"),
                     stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for _ in range(30):
        if ollama_up(): break
        time.sleep(2)
profile = json.load(open(LEGAL / "config/domains.json"))
subprocess.run(["ollama", "pull", profile["legal"]["model"]], check=True)
subprocess.run(["ollama", "pull", OCR_MODEL], check=True)

# keys from the Kaggle secrets, as the legal notebook reads them
from kaggle_secrets import UserSecretsClient
secrets, keys = UserSecretsClient(), []
for name in ["GEMINI_API_KEY"] + [f"GEMINI_API_KEY_{i}" for i in range(2, 10)]:
    try:
        keys.append(secrets.get_secret(name))
    except Exception:
        pass
if not keys:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets) and attach it.")

# the legal profile: answers on Ollama unless ANSWERS = "gemini"; optional fine-tuned Agent 1
if ANSWERS == "ollama":
    profile["legal"]["answer_backend"] = "same"
if AGENT1_ADAPTER:
    profile["legal"].update(agent1_backend="transformers", agent1_adapter=AGENT1_ADAPTER)
    os.environ["SANITISE_AGENT1_DEVICE"] = "cuda:1"
json.dump(profile, open(LEGAL / "config/domains.json", "w"), indent=2)
print(json.dumps(profile["legal"], indent=1))

WORK = W / "eval" / ("legal" if SCALE == "full" else f"legal_{SCALE}")
os.environ.update({
    "SANITISE_DOMAIN": "legal", "SANITISE_LLM_BACKEND": "ollama", "SANITISE_LLM_TIMEOUT_S": "300",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "GEMINI_API_KEYS": ",".join(keys), "SANITISE_GEMINI_CACHE_DIR": str(W / "gemini_cache"),
    "SANITISE_GEMINI_RPM": "10", "SANITISE_GEMINI_KEY_COOLDOWN_S": "60",
})
SUITE = MAIN / "tools/eval_suite/run.py"
def run(step, extra="", repo=LEGAL):
    cmd = (f"python {SUITE} {step} --domain legal --work {WORK} --scale {SCALE} "
           f"--repo {repo} --main-repo {MAIN} {extra}")
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)
print(f"ready: legal pipeline {LEGAL_BRANCH} | {len(keys)} Gemini key(s) | OCR {OCR_MODEL} | "
      f"results in {WORK}")

In [ ]:
# 2 Data: real public text downloaded now, the generated corpus copied, every question written
run("data")

In [ ]:
# 3 End to end, part 1: the 50 real scans read by OCR (main's step 0), cut into parts
# (main's code reads the scans; the legal agents take the parts in cell 5)
run("e2e-prep", repo=MAIN)

In [ ]:
# 4 End to end, part 2: Gemini marks each scan's private values and writes 2 questions after them
# and 2 about the content (saved in e2e/labels.json: open it and check a few, see the guide)
run("e2e-label")

In [ ]:
# 5 Build: Agents 1-3 over the generated corpus, the real corpus and both end-to-end sets
# (long; re-run after a time limit: documents already in a store are skipped)
run("build")

In [ ]:
# 6 Single-question attacks: Target Info, Repeat / ROUGE Prompt, membership inference, utility
run("single")

In [ ]:
# 7 Multi-question attacks: 180 sessions of 8 rounds, no privacy vs ours (with the monitor)
run("multi")

In [ ]:
# 8 End to end, part 3: the scans' questions on the gold-text store (A) and the OCR store (B)
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_legal{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/utility.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"legal/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")